# 🚪 MOCG Capa de Movimiento — Badges y Espacios
**Detección por manifold espaciotemporal con privacidad por diseño**

## Arquitectura de dos capas
```
CAPA ANÓNIMA (siempre activa)
  Tokens pseudoanónimos por badge (HMAC-SHA256 + sal bimestral)
  Manifold del badge: patrón de movimiento del token
  Manifold del espacio: densidad de ocupación por zona
  Detección de anomalías sin identidad

CAPA DE IDENTIDAD (requiere autorización dual)
  Resolución token -> persona real
  Activada SOLO cuando score supera umbral sostenido
  Requiere dos administradores distintos
  Auditable: cada resolución queda en el log forense
```

## Principio de privacidad
El motor nunca necesita saber quién es la persona.
La sal rota bimestralmente: tokens de B1 != tokens de B2.

## Lo que solo este módulo detecta
- Imposible travel: mismo token en dos lugares simultáneos
- Secuencia inversa: badge en destino antes que en entrada
- Espacio anómalo: zona con ocupación fuera del manifold
- Badge clonado: dos instancias del mismo token activas
- Patrón de rol roto: movimiento inconsistente con el rol

## Integración MOCG
Eventos `movement_anomaly` al PolicyAdapter del Modo 2.
Mismo contrato que Resistor, Nodo Físico y Capa Administrativa.

## 1. Instalación de dependencias

In [ ]:
!pip install gradio numpy pandas matplotlib plotly networkx requests -q
print('✅ Dependencias listas.')

## 2. Configuración, topología del edificio y tokenización

El grafo del edificio define qué espacios conectan físicamente
y el tiempo mínimo de tránsito entre ellos.
La topología es la base del detector de imposible travel.

In [ ]:
import json, uuid, hashlib, time, copy, hmac
import numpy as np
import pandas as pd
import networkx as nx
from pathlib import Path
from datetime import datetime, timezone, timedelta
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Set, Any
from collections import defaultdict, deque

DEFAULT_CONFIG = {
    'org_name':     'Demo MOCG Movimiento',
    'principal_id': 'ORG-DEMO-001',
    'bimester':     '2025-B1',
    'token_salt':   'mocg-salt-2025-B1',
    'mode2_webhook_url': '',
    'spaces': [
        {'id': 'entrada',      'name': 'Entrada principal',    'type': 'transit',    'capacity': 50, 'coords': [0, 0],  'sensitivity': 'low'},
        {'id': 'lobby',        'name': 'Lobby',                'type': 'common',     'capacity': 30, 'coords': [1, 0],  'sensitivity': 'low'},
        {'id': 'oficinas_a',   'name': 'Oficinas Ala A',       'type': 'office',     'capacity': 20, 'coords': [2, 1],  'sensitivity': 'medium'},
        {'id': 'oficinas_b',   'name': 'Oficinas Ala B',       'type': 'office',     'capacity': 20, 'coords': [2, -1], 'sensitivity': 'medium'},
        {'id': 'sala_reuniones','name': 'Sala de reuniones',   'type': 'meeting',    'capacity': 12, 'coords': [3, 0],  'sensitivity': 'medium'},
        {'id': 'servidores',   'name': 'Sala de servidores',   'type': 'critical',   'capacity': 4,  'coords': [4, 1],  'sensitivity': 'critical'},
        {'id': 'archivo',      'name': 'Archivo confidencial', 'type': 'restricted', 'capacity': 3,  'coords': [4, -1], 'sensitivity': 'high'},
        {'id': 'salida',       'name': 'Salida',               'type': 'transit',    'capacity': 50, 'coords': [5, 0],  'sensitivity': 'low'},
    ],
    'adjacency': [
        ['entrada',     'lobby',          30],
        ['lobby',       'oficinas_a',     60],
        ['lobby',       'oficinas_b',     60],
        ['lobby',       'sala_reuniones', 90],
        ['oficinas_a',  'servidores',    120],
        ['oficinas_b',  'archivo',        90],
        ['sala_reuniones', 'servidores',  60],
        ['oficinas_a',  'salida',        120],
        ['oficinas_b',  'salida',        120],
        ['lobby',       'salida',        180],
    ],
    'role_permissions': {
        'admin':    ['entrada','lobby','oficinas_a','oficinas_b','sala_reuniones','servidores','archivo','salida'],
        'staff':    ['entrada','lobby','oficinas_a','oficinas_b','sala_reuniones','salida'],
        'external': ['entrada','lobby','sala_reuniones','salida'],
        'it':       ['entrada','lobby','oficinas_a','servidores','salida'],
        'legal':    ['entrada','lobby','oficinas_b','archivo','salida'],
        'visitor':  ['entrada','lobby','salida'],
    },
    'badge_roles': {
        'B001': 'admin', 'B002': 'staff', 'B003': 'staff',
        'B004': 'external', 'B005': 'it', 'B006': 'legal',
        'B007': 'visitor', 'B008': 'staff',
    },
    'thresholds': {
        'badge_dissonance_alert':    0.30,
        'space_dissonance_alert':    0.35,
        'combined_dissonance_alert': 0.40,
        'combined_dissonance_id':    0.65,
        'id_resolution_events':      3,
        'impossible_travel_factor':  1.2,
        'space_occupancy_sigma':     2.5,
    }
}


def load_config(path='movement_config.json'):
    p = Path(path)
    if p.exists():
        with open(p) as f:
            custom = json.load(f)
        merged = {**DEFAULT_CONFIG, **custom}
        for k in ('thresholds','role_permissions','badge_roles'):
            if k in DEFAULT_CONFIG and k in custom:
                merged[k] = {**DEFAULT_CONFIG[k], **custom[k]}
        print(f'Config cargada: {merged["org_name"]}')
    else:
        merged = copy.deepcopy(DEFAULT_CONFIG)
        print(f'Sin movement_config.json, usando defaults')
    return merged


MCFG = load_config()
RUN_ID = f"MOCG-MOV-{MCFG['principal_id']}-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')}"
ARTIFACTS = Path('mocg_movement_artifacts') / RUN_ID
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Grafo del edificio
BUILDING_GRAPH = nx.Graph()
SPACE_MAP = {s['id']: s for s in MCFG['spaces']}
for space in MCFG['spaces']:
    BUILDING_GRAPH.add_node(space['id'], **space)
for src, dst, transit_s in MCFG['adjacency']:
    BUILDING_GRAPH.add_edge(src, dst, transit_seconds=transit_s)


# --- Tokenizacion anonima ---
def tokenize_badge(badge_id, salt, length=12):
    raw = hmac.new(salt.encode(), badge_id.encode(), hashlib.sha256).hexdigest()
    return f'TK-{raw[:length].upper()}'


def resolve_token(token, salt, registry):
    for badge_id in registry:
        if tokenize_badge(badge_id, salt) == token:
            return badge_id
    return None


TOKEN_MAP = {}   # token -> role
REVERSE_MAP = {} # badge_id -> token  (solo para el generador)
salt = MCFG['token_salt']
for badge_id, role in MCFG['badge_roles'].items():
    token = tokenize_badge(badge_id, salt)
    TOKEN_MAP[token] = role
    REVERSE_MAP[badge_id] = token

print(f'Run ID    : {RUN_ID}')
print(f'Espacios  : {len(MCFG["spaces"])}')
print(f'Conexiones: {BUILDING_GRAPH.number_of_edges()}')
print(f'Badges    : {len(MCFG["badge_roles"])}')
print(f'Tokens generados (capa anonima):')
for tk, role in list(TOKEN_MAP.items())[:4]:
    print(f'  {tk} -> {role}')
print('  ...')

## 3. Estructuras de datos y generador de movimientos

In [ ]:
@dataclass
class MovementEvent:
    event_id:      str
    timestamp:     float
    token:         str
    token_role:    str
    space_id:      str
    direction:     str
    access_granted:bool
    reader_id:     str
    _badge_id:     str = ''
    _anomaly:      str = ''


def generate_events(cfg, n_days=20, scenario='normal', seed=42):
    import random
    rng = random.Random(seed)
    base = datetime(2025, 1, 20, 0, 0, 0, tzinfo=timezone.utc)
    role_perms = cfg['role_permissions']
    events = []

    def min_transit(src, dst):
        try:
            path = nx.shortest_path(BUILDING_GRAPH, src, dst, weight='transit_seconds')
            return sum(BUILDING_GRAPH[path[i]][path[i+1]]['transit_seconds'] for i in range(len(path)-1))
        except:
            return 300

    def typical_seq(role):
        perms = role_perms.get(role, ['entrada','lobby','salida'])
        work = [s for s in perms if s not in ('entrada','lobby','salida')]
        seq = ['entrada','lobby']
        if work:
            seq += rng.sample(work, min(len(work), 2))
        seq += ['salida']
        return seq

    def make_event(token, role, space, direction, ts, granted=True, badge_id='', anomaly=''):
        return MovementEvent(
            event_id=str(uuid.uuid4())[:8],
            timestamp=ts, token=token, token_role=role,
            space_id=space, direction=direction,
            access_granted=granted,
            reader_id=f'RDR-{space[:3].upper()}-{rng.randint(1,3):02d}',
            _badge_id=badge_id, _anomaly=anomaly,
        )

    # Trafico normal
    n_normal = n_days if scenario == 'normal' else n_days // 2
    for day in range(n_normal * 5):  # 5 badges x dias
        badge_id = rng.choice(list(cfg['badge_roles'].keys()))
        role = cfg['badge_roles'][badge_id]
        token = REVERSE_MAP[badge_id]
        dt = base + timedelta(days=day // 5)
        if dt.weekday() >= 5 and role not in ('admin','it'):
            if rng.random() > 0.05:
                continue
        arrive_h = rng.randint(8, 10)
        ts = dt.replace(hour=arrive_h, minute=rng.randint(0, 59)).timestamp()
        seq = typical_seq(role)
        for i, space in enumerate(seq):
            events.append(make_event(token, role, space, 'enter', ts, badge_id=badge_id))
            stay = rng.randint(5, 30) * 60
            events.append(make_event(token, role, space, 'exit', ts + stay, badge_id=badge_id))
            if i < len(seq) - 1:
                transit = min_transit(space, seq[i+1])
                ts = ts + stay + transit + rng.randint(0, 60)

    # Ataques
    atk_badge = 'B004'
    atk_token = REVERSE_MAP[atk_badge]
    atk_role  = cfg['badge_roles'][atk_badge]
    atk_ts    = (base + timedelta(days=n_days//2, hours=14, minutes=23)).timestamp()

    if scenario in ('impossible_travel', 'mixed'):
        events.append(make_event(atk_token, atk_role, 'lobby', 'enter', atk_ts,
                                  badge_id=atk_badge, anomaly='impossible_travel'))
        # 45s despues en servidores (minimo 210s)
        events.append(make_event(atk_token, atk_role, 'servidores', 'enter', atk_ts+45,
                                  badge_id=atk_badge, anomaly='impossible_travel'))

    if scenario in ('role_violation', 'mixed'):
        events.append(make_event(atk_token, atk_role, 'servidores', 'enter',
                                  atk_ts+3600, granted=False,
                                  badge_id=atk_badge, anomaly='role_violation'))
        events.append(make_event(atk_token, atk_role, 'archivo', 'enter',
                                  atk_ts+3700, granted=False,
                                  badge_id=atk_badge, anomaly='role_violation'))

    if scenario in ('space_anomaly', 'mixed'):
        anomaly_ts = atk_ts + 7200
        for i, (bid, role) in enumerate(list(cfg['badge_roles'].items())[:8]):
            tk = REVERSE_MAP[bid]
            events.append(make_event(tk, role, 'servidores', 'enter',
                                      anomaly_ts + i*30,
                                      badge_id=bid, anomaly='space_anomaly'))

    events.sort(key=lambda e: e.timestamp)
    print(f'Generados: {len(events)} eventos | escenario: {scenario}')
    return events


print('Estructuras y generador listos.')

## 4. Motor de manifold y detección de anomalías

In [ ]:
def signature_badge(events, token, cfg, baseline=None):
    token_evts = [e for e in events if e.token == token]
    if len(token_evts) < 2:
        return np.full(7, 0.1, dtype=np.float32)
    role = TOKEN_MAP.get(token, 'staff')
    role_perms = set(cfg['role_permissions'].get(role, []))
    sorted_evts = sorted(token_evts, key=lambda e: e.timestamp)

    off = sum(1 for e in token_evts
              if not (8 <= datetime.fromtimestamp(e.timestamp, tz=timezone.utc).hour <= 20))
    off_r = float(off / len(token_evts))

    spaces = set(e.space_id for e in token_evts)
    sp_div = float(len(spaces) / max(len(MCFG['spaces']), 1))

    crit_spaces = {s['id'] for s in cfg['spaces'] if s['sensitivity'] in ('critical','high')}
    crit_r = float(sum(1 for e in token_evts if e.space_id in crit_spaces) / max(len(token_evts), 1))

    denied_r = float(sum(1 for e in token_evts if not e.access_granted) / max(len(token_evts), 1))

    # Velocity anomaly (impossible travel)
    vel_violations = 0
    for i in range(1, len(sorted_evts)):
        prev, curr = sorted_evts[i-1], sorted_evts[i]
        if prev.space_id != curr.space_id:
            try:
                path = nx.shortest_path(BUILDING_GRAPH, prev.space_id, curr.space_id, weight='transit_seconds')
                min_t = sum(BUILDING_GRAPH[path[j]][path[j+1]]['transit_seconds'] for j in range(len(path)-1))
            except:
                min_t = 300
            factor = cfg['thresholds']['impossible_travel_factor']
            if curr.timestamp - prev.timestamp < min_t / factor:
                vel_violations += 1
    vel_anom = float(min(vel_violations / max(len(sorted_evts)-1, 1), 1.0))

    space_seq = [e.space_id for e in sorted_evts]
    space_counts = defaultdict(int)
    for s in space_seq:
        space_counts[s] += 1
    probs = np.array(list(space_counts.values()), dtype=float)
    probs /= probs.sum()
    entropy = float(-np.sum(probs * np.log2(probs + 1e-10)))
    max_ent = np.log2(max(len(space_counts), 2))
    seq_reg = 1.0 - float(entropy / max_ent)

    ts_list = [e.timestamp for e in token_evts]
    span_days = max((max(ts_list) - min(ts_list)) / 86400, 1)
    freq = float(min(len(token_evts) / span_days / 20, 1.0))

    return np.array([np.clip(v, 0, 1) for v in
        [off_r, sp_div, crit_r, denied_r, vel_anom, seq_reg, freq]],
        dtype=np.float32)


def signature_space(events, space_id, cfg, window_h=1.0):
    space_cfg = SPACE_MAP.get(space_id, {})
    capacity = space_cfg.get('capacity', 10)
    space_evts = sorted([e for e in events if e.space_id == space_id],
                         key=lambda e: e.timestamp)
    if not space_evts:
        return {'series':[], 'mean':0, 'std':1, 'capacity':capacity, 'space_id':space_id}

    ts_min = space_evts[0].timestamp
    ts_max = space_evts[-1].timestamp
    win_s = window_h * 3600
    series = []
    current = ts_min
    while current < ts_max:
        in_window = set()
        for e in space_evts:
            if current <= e.timestamp < current + win_s:
                if e.direction == 'enter' and e.access_granted:
                    in_window.add(e.token)
                elif e.direction == 'exit':
                    in_window.discard(e.token)
        series.append({'ts': current, 'occupancy': len(in_window), 'tokens': list(in_window)})
        current += win_s

    occ_vals = [o['occupancy'] for o in series]
    return {'series': series, 'mean': float(np.mean(occ_vals)),
            'std': float(np.std(occ_vals)), 'capacity': capacity, 'space_id': space_id}


def detect_impossible_travel(events, token, cfg):
    token_evts = sorted([e for e in events if e.token == token], key=lambda e: e.timestamp)
    violations = []
    factor = cfg['thresholds']['impossible_travel_factor']
    for i in range(1, len(token_evts)):
        prev, curr = token_evts[i-1], token_evts[i]
        if prev.space_id == curr.space_id:
            continue
        elapsed = curr.timestamp - prev.timestamp
        try:
            path = nx.shortest_path(BUILDING_GRAPH, prev.space_id, curr.space_id, weight='transit_seconds')
            min_t = sum(BUILDING_GRAPH[path[j]][path[j+1]]['transit_seconds'] for j in range(len(path)-1))
        except:
            min_t = 600
        if elapsed < min_t / factor:
            violations.append((curr.timestamp,
                f'Imposible travel: {prev.space_id}->{curr.space_id} en {elapsed:.0f}s (min={min_t:.0f}s)',
                curr.space_id))
    return violations


print('Motor de manifold y detectores listos.')

## 5. Pipeline de análisis y sistema de resolución de identidad

In [ ]:
@dataclass
class MovementRecord:
    record_id:           str
    timestamp:           str
    run_id:              str
    token:               str
    token_role:          str
    space_id:            str
    anomaly_type:        str
    badge_dissonance:    float
    space_dissonance:    float
    combined_dissonance: float
    description:         str
    id_resolution_requested: bool = False
    id_resolution_authorized: bool = False
    id_resolution_by:    List[str] = field(default_factory=list)
    resolved_badge_id:   str = ''
    action:              str = ''
    override_token:      str = ''
    sha256:              str = ''

    def compute_hash(self):
        payload = json.dumps({'record_id': self.record_id,
            'token': self.token, 'anomaly': self.anomaly_type,
            'combined': round(self.combined_dissonance, 4)}, sort_keys=True)
        return hashlib.sha256(payload.encode()).hexdigest()[:16]

    def to_mode2_payload(self):
        return {'event': 'movement_anomaly', 'record_id': self.record_id,
            'timestamp': self.timestamp, 'run_id': self.run_id,
            'entity_id': self.token, 'action': self.action,
            'dissonance': round(self.combined_dissonance, 4),
            'executed': False, 'reversible': True,
            'override_token': self.override_token, 'ttl_minutes': 30,
            'metadata': {'token_role': self.token_role,
                'space_id': self.space_id, 'anomaly_type': self.anomaly_type,
                'description': self.description,
                'id_resolution_req': self.id_resolution_requested},
            'sha256': self.sha256}


FORENSIC_LOG: List[MovementRecord] = []
_token_alert_count: Dict[str, int] = defaultdict(int)
_badge_baselines:   Dict[str, np.ndarray] = {}
ID_RESOLUTION_LOG:  List[dict] = []


def run_movement_analysis(events, cfg, dry_run=True):
    global FORENSIC_LOG, _token_alert_count, _badge_baselines
    FORENSIC_LOG = []
    _token_alert_count = defaultdict(int)
    _badge_baselines = {}

    th = cfg['thresholds']
    tokens = list(set(e.token for e in events))
    spaces = [s['id'] for s in cfg['spaces']]

    # Baselines (primera mitad)
    sorted_evts = sorted(events, key=lambda e: e.timestamp)
    split = max(len(sorted_evts)//2, 10)
    baseline_evts = sorted_evts[:split]
    for token in tokens:
        _badge_baselines[token] = signature_badge(baseline_evts, token, cfg)

    # Manifolds de espacios
    space_manifolds = {sid: signature_space(events, sid, cfg) for sid in spaces}

    # Anomalias de espacio
    space_anomalies = {}
    sigma_th = th['space_occupancy_sigma']
    for sid, manifold in space_manifolds.items():
        mean, std, cap = manifold['mean'], max(manifold['std'], 0.1), manifold['capacity']
        anoms = []
        for window in manifold['series']:
            occ = window['occupancy']
            z   = (occ - mean) / std
            if abs(z) > sigma_th:
                score = float(min(abs(z) / (sigma_th*2), 1.0))
                anoms.append((window['ts'], score,
                    f'Ocupacion {occ} (sigma={z:.1f}) normal={mean:.1f}+-{std:.1f} cap={cap}'))
            if occ > cap:
                anoms.append((window['ts'], min(occ/cap, 1.0), f'Sobre capacidad: {occ}/{cap}'))
        if anoms:
            space_anomalies[sid] = anoms

    # Anomalias de badge
    for token in tokens:
        role = TOKEN_MAP.get(token, 'staff')
        baseline = _badge_baselines.get(token, np.full(7, 0.1))
        curr_vec = signature_badge(events, token, cfg, baseline)
        badge_dis = float(np.mean(np.abs(curr_vec - baseline)))
        if badge_dis < th['badge_dissonance_alert']:
            _badge_baselines[token] = 0.95*baseline + 0.05*curr_vec

        it_violations = detect_impossible_travel(events, token, cfg)
        role_perms = set(cfg['role_permissions'].get(role, []))
        role_viols = [e for e in events if e.token == token and e.space_id not in role_perms]

        badge_dis_amp = badge_dis
        anomaly_type = 'behavioral'
        description = ''
        if it_violations:
            badge_dis_amp = min(badge_dis_amp * 3.0, 1.0)
            anomaly_type = 'impossible_travel'
            description = it_violations[0][1]
        if role_viols:
            badge_dis_amp = min(badge_dis_amp + 0.3, 1.0)
            anomaly_type = ('role_violation' if not it_violations
                            else 'impossible_travel+role_violation')
            if not description:
                description = f'{len(role_viols)} accesos fuera de permisos del rol {role}'

        token_spaces = set(e.space_id for e in events if e.token == token)
        space_dis = 0.0
        for sid in token_spaces:
            if sid in space_anomalies:
                space_dis = max(space_dis, max(s for _, s, _ in space_anomalies[sid]))

        combined = badge_dis_amp * 0.6 + space_dis * 0.4
        if badge_dis_amp > th['badge_dissonance_alert'] and space_dis > 0:
            combined = min(combined * 1.5, 1.0)

        if not description and space_dis > 0:
            description = f'Token en espacio anomalo'

        if combined < th['combined_dissonance_alert'] and not it_violations:
            continue

        _token_alert_count[token] += 1
        id_req = (combined >= th['combined_dissonance_id'] and
                  _token_alert_count[token] >= th['id_resolution_events'])

        action = ('request_id_resolution' if id_req
                  else 'report_impossible_travel' if it_violations
                  else 'report_movement_anomaly')

        trigger_evts = sorted([e for e in events if e.token == token], key=lambda e: e.timestamp)
        trigger_ts   = trigger_evts[-1].timestamp if trigger_evts else time.time()
        trigger_space = trigger_evts[-1].space_id if trigger_evts else ''
        ts_str = datetime.fromtimestamp(trigger_ts, tz=timezone.utc).isoformat()

        rec = MovementRecord(
            record_id=f'MR-{str(uuid.uuid4())[:8].upper()}',
            timestamp=ts_str, run_id=RUN_ID,
            token=token, token_role=role,
            space_id=trigger_space, anomaly_type=anomaly_type,
            badge_dissonance=round(badge_dis_amp, 4),
            space_dissonance=round(space_dis, 4),
            combined_dissonance=round(combined, 4),
            description=description,
            id_resolution_requested=id_req,
            action=action,
            override_token=str(uuid.uuid4())[:12],
        )
        rec.sha256 = rec.compute_hash()
        FORENSIC_LOG.append(rec)

    # Anomalias de espacio sin token
    for sid, anoms in space_anomalies.items():
        for ts_a, score_a, desc_a in anoms:
            ts_str = datetime.fromtimestamp(ts_a, tz=timezone.utc).isoformat()
            rec = MovementRecord(
                record_id=f'MR-{str(uuid.uuid4())[:8].upper()}',
                timestamp=ts_str, run_id=RUN_ID,
                token='SPACE_LEVEL', token_role='space',
                space_id=sid, anomaly_type='space_occupancy',
                badge_dissonance=0.0, space_dissonance=round(score_a, 4),
                combined_dissonance=round(score_a, 4),
                description=desc_a, action='report_space_anomaly',
                override_token=str(uuid.uuid4())[:12],
            )
            rec.sha256 = rec.compute_hash()
            FORENSIC_LOG.append(rec)

    id_reqs = sum(1 for r in FORENSIC_LOG if r.id_resolution_requested)
    it_det  = sum(1 for r in FORENSIC_LOG if 'impossible_travel' in r.anomaly_type)
    print(f'Analisis completado: {len(FORENSIC_LOG)} registros | IT: {it_det} | ID req: {id_reqs}')
    return {'records': FORENSIC_LOG, 'space_manifolds': space_manifolds,
            'space_anomalies': space_anomalies, 'events': events,
            'stats': {'total_events': len(events), 'forensic_recs': len(FORENSIC_LOG),
                      'impossible_travel': it_det, 'id_requests': id_reqs, 'tokens': tokens}}


def request_id_resolution(record_id, auth1, auth2, cfg):
    if auth1 == auth2:
        return {'ok': False, 'msg': 'Requiere dos autorizadores distintos'}
    rec = next((r for r in FORENSIC_LOG if r.record_id == record_id), None)
    if not rec:
        return {'ok': False, 'msg': f'Registro no encontrado: {record_id}'}
    if not rec.id_resolution_requested:
        return {'ok': False, 'msg': 'Este registro no requiere resolucion de identidad'}
    badge_id = resolve_token(rec.token, cfg['token_salt'], cfg['badge_roles'])
    if not badge_id:
        return {'ok': False, 'msg': 'No se pudo resolver el token'}
    rec.id_resolution_authorized = True
    rec.id_resolution_by = [auth1, auth2]
    rec.resolved_badge_id = badge_id
    ID_RESOLUTION_LOG.append({'ts': datetime.now(timezone.utc).isoformat(),
        'record_id': record_id, 'token': rec.token, 'resolved_to': badge_id,
        'authorized_by': [auth1, auth2], 'dissonance': rec.combined_dissonance})
    return {'ok': True, 'badge_id': badge_id,
            'role': cfg['badge_roles'].get(badge_id, 'unknown'),
            'msg': f'Identidad resuelta: {badge_id} | autorizadores: {auth1}, {auth2}'}


# Demo
print('Ejecutando escenario mixed...')
DEMO_EVENTS = generate_events(MCFG, n_days=20, scenario='mixed', seed=42)
DEMO_RESULT = run_movement_analysis(DEMO_EVENTS, MCFG, dry_run=True)

## 6. Visualizaciones

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from PIL import Image
import io as io_mod
from IPython.display import display
pio.renderers.default = 'notebook'


def plot_token_timeline(events, records, cfg):
    tokens = list(set(e.token for e in events))[:6]
    palette = ['#378ADD','#639922','#BA7517','#E24B4A','#9B59B6','#D85A30']
    space_ids = [s['id'] for s in cfg['spaces']]
    anom_tokens = {r.token for r in records if r.anomaly_type != 'space_occupancy'}
    fig = go.Figure()
    for i, token in enumerate(tokens):
        t_evts = sorted([e for e in events if e.token == token], key=lambda e: e.timestamp)
        role = TOKEN_MAP.get(token, '?')
        col  = palette[i % len(palette)]
        xs   = [datetime.fromtimestamp(e.timestamp, tz=timezone.utc) for e in t_evts]
        ys   = [space_ids.index(e.space_id) if e.space_id in space_ids else -1 for e in t_evts]
        txt  = [f'{e.space_id} ({e.direction})' for e in t_evts]
        fig.add_trace(go.Scatter(x=xs, y=ys, mode='lines+markers',
            name=f'{token[:8]} [{role}]',
            line=dict(color=col, width=1.5),
            marker=dict(size=6, color=['#E24B4A' if e.token in anom_tokens else col for e in t_evts]),
            text=txt, hovertemplate='%{text}<extra>%{fullData.name}</extra>'))
    fig.update_layout(title='Trayectorias de tokens',
        xaxis_title='Tiempo',
        yaxis=dict(tickmode='array', tickvals=list(range(len(space_ids))),
                   ticktext=space_ids, title='Espacio'),
        height=380, margin=dict(l=120,r=40,t=60,b=40),
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        font=dict(family='sans-serif',size=10), yaxis_gridcolor='#e8e8e4',
        legend=dict(orientation='h',y=-0.2))
    return fig


def plot_space_occupancy(space_manifolds, space_anomalies, cfg):
    palette = ['#378ADD','#639922','#BA7517','#E24B4A','#9B59B6','#D85A30','#1ABC9C','#F39C12']
    fig = go.Figure()
    for i, (sid, manifold) in enumerate(space_manifolds.items()):
        series = manifold['series']
        if not series:
            continue
        xs  = [datetime.fromtimestamp(s['ts'], tz=timezone.utc) for s in series]
        ys  = [s['occupancy'] for s in series]
        col = palette[i % len(palette)]
        fig.add_trace(go.Scatter(x=xs, y=ys, mode='lines', name=sid,
            line=dict(color=col, width=1.5)))
        if sid in space_anomalies:
            ax_anom = [datetime.fromtimestamp(ts_a, tz=timezone.utc) for ts_a, _, _ in space_anomalies[sid]]
            ay_anom = []
            for ts_a, _, _ in space_anomalies[sid]:
                occ = next((s['occupancy'] for s in series if abs(s['ts']-ts_a) < 1800), 0)
                ay_anom.append(occ)
            fig.add_trace(go.Scatter(x=ax_anom, y=ay_anom, mode='markers',
                name=f'{sid} anomalia', showlegend=False,
                marker=dict(color='#E24B4A', size=10, symbol='triangle-up')))
        fig.add_hline(y=manifold['capacity'], line_dash='dot',
            line_color=col, line_width=0.8,
            annotation_text=f'cap {sid}', annotation_font_size=7)
    fig.update_layout(title='Ocupacion por espacio',
        xaxis_title='Tiempo', yaxis_title='Tokens en espacio',
        height=340, margin=dict(l=40,r=40,t=60,b=40),
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        font=dict(family='sans-serif',size=10), yaxis_gridcolor='#e8e8e4',
        legend=dict(orientation='h',y=-0.2))
    return fig


def plot_building_map(result, cfg):
    fig, ax = plt.subplots(figsize=(8,5), dpi=100)
    fig.patch.set_facecolor('#f8f8f6')
    ax.set_facecolor('#f8f8f6')
    spaces = {s['id']: s for s in cfg['spaces']}
    anomalous = set(r.space_id for r in result['records'] if r.space_id)
    sens_colors = {'low':'#B5D4F4','medium':'#C0DD97','high':'#FAC775','critical':'#F7C1C1'}
    for src, dst, _ in cfg['adjacency']:
        if src in spaces and dst in spaces:
            sx, sy = spaces[src]['coords']
            dx, dy = spaces[dst]['coords']
            ax.plot([sx,dx],[sy,dy], color='#D3D1C7', linewidth=1.5, zorder=1, alpha=0.7)
    for sid, space in spaces.items():
        cx, cy = space['coords']
        col  = sens_colors.get(space['sensitivity'],'#B5D4F4')
        edge = '#E24B4A' if sid in anomalous else '#9c9a92'
        lw   = 2.5 if sid in anomalous else 1.0
        circle = plt.Circle((cx,cy), 0.35, color=col, ec=edge, lw=lw, zorder=2)
        ax.add_patch(circle)
        ax.text(cx, cy, sid.replace('_','\n'), ha='center', va='center',
                fontsize=6.5, fontweight='bold', color='#2C2C2A', zorder=5)
    handles = [mpatches.Patch(color=v, label=k) for k, v in sens_colors.items()]
    handles.append(mpatches.Patch(facecolor='white', edgecolor='#E24B4A',
                                   linewidth=2, label='Espacio anomalo'))
    ax.legend(handles=handles, loc='lower right', fontsize=7, framealpha=0.8)
    ax.set_xlim(-0.8, 5.8)
    ax.set_ylim(-1.8, 1.5)
    ax.set_title('Mapa del edificio', fontsize=10)
    ax.axis('off')
    plt.tight_layout()
    buf = io_mod.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).copy()


if DEMO_RESULT:
    plot_token_timeline(DEMO_RESULT['events'], DEMO_RESULT['records'], MCFG).show()
    plot_space_occupancy(DEMO_RESULT['space_manifolds'], DEMO_RESULT['space_anomalies'], MCFG).show()
    display(plot_building_map(DEMO_RESULT, MCFG))

## 7. Dashboard Gradio

In [ ]:
import gradio as gr

_last_result = DEMO_RESULT

SCENARIO_LABELS = {
    'normal':           'Normal - flujos rutinarios',
    'impossible_travel':'Imposible travel',
    'role_violation':   'Violacion de rol',
    'space_anomaly':    'Anomalia de espacio',
    'mixed':            'Mixto - multiples anomalias',
}


def run_dashboard(scenario, n_days, dry_run_chk, seed, cfg_json):
    global MCFG, BUILDING_GRAPH, SPACE_MAP, TOKEN_MAP, REVERSE_MAP, _last_result
    try:
        custom = json.loads(cfg_json) if cfg_json.strip() else {}
        MCFG = {**DEFAULT_CONFIG, **custom}
        for k in ('thresholds','role_permissions','badge_roles'):
            if k in DEFAULT_CONFIG and k in custom:
                MCFG[k] = {**DEFAULT_CONFIG[k], **custom[k]}
    except Exception as e:
        return (None, None, None, str(e), '', '', '', '', '', [])

    BUILDING_GRAPH = nx.Graph()
    SPACE_MAP = {s['id']: s for s in MCFG['spaces']}
    for space in MCFG['spaces']:
        BUILDING_GRAPH.add_node(space['id'], **space)
    for src, dst, t in MCFG['adjacency']:
        BUILDING_GRAPH.add_edge(src, dst, transit_seconds=t)
    TOKEN_MAP.clear()
    REVERSE_MAP.clear()
    salt = MCFG['token_salt']
    for bid, role in MCFG['badge_roles'].items():
        tk = tokenize_badge(bid, salt)
        TOKEN_MAP[tk] = role
        REVERSE_MAP[bid] = tk

    events = generate_events(MCFG, n_days=int(n_days), scenario=scenario, seed=int(seed))
    result = run_movement_analysis(events, MCFG, dry_run=dry_run_chk)
    _last_result = result

    fig_tl   = plot_token_timeline(events, result['records'], MCFG)
    fig_occ  = plot_space_occupancy(result['space_manifolds'], result['space_anomalies'], MCFG)
    fig_bldg = plot_building_map(result, MCFG)

    s = result['stats']
    table = []
    for r in result['records'][:50]:
        icon = ('ID' if r.id_resolution_requested
                else ('IT' if 'impossible_travel' in r.anomaly_type else 'W'))
        table.append([r.record_id, r.token[:12], r.token_role, r.space_id,
                      r.anomaly_type, f'{r.combined_dissonance:.3f}',
                      icon, r.description[:60] or '-', r.override_token])

    return (fig_tl, fig_occ, fig_bldg,
            str(s['total_events']), str(s['forensic_recs']),
            str(s['impossible_travel']), str(s['id_requests']),
            str(len(s['tokens'])), '', table)


def do_id_resolution(record_id, auth1, auth2):
    result = request_id_resolution(record_id.strip(), auth1.strip(), auth2.strip(), MCFG)
    return result['msg']


DEFAULT_CFG_UI = json.dumps({
    'org_name':   MCFG['org_name'],
    'bimester':   MCFG['bimester'],
    'token_salt': MCFG['token_salt'],
    'thresholds': MCFG['thresholds'],
    'mode2_webhook_url': '',
}, indent=2)


with gr.Blocks(title='MOCG Capa de Movimiento',
               theme=gr.themes.Soft(primary_hue='blue', neutral_hue='slate')) as demo:

    gr.Markdown(f'''
# MOCG Capa de Movimiento - Badges y Espacios
**{MCFG["org_name"]}** Bimestre {MCFG["bimester"]}

Deteccion por manifold espaciotemporal con privacidad por diseno.
Tokens pseudoanonimos - la identidad se revela solo con autorizacion dual.
''')

    with gr.Row():
        with gr.Column(scale=2):
            cfg_box = gr.Code(value=DEFAULT_CFG_UI, language='json',
                              label='movement_config', lines=10)
            with gr.Row():
                scenario_dd = gr.Dropdown(choices=list(SCENARIO_LABELS.keys()),
                                          value='mixed', label='Escenario')
                n_days_sl   = gr.Slider(5, 60, value=20, step=5, label='Dias')
            with gr.Row():
                dry_run_chk = gr.Checkbox(value=True, label='dry_run')
                seed_sl     = gr.Slider(1, 999, value=42, step=1, label='Semilla')
            run_btn = gr.Button('Ejecutar analisis', variant='primary', size='lg')
        with gr.Column(scale=1):
            gr.Markdown('### Metricas')
            with gr.Row():
                m_evts = gr.Textbox(label='Eventos',    interactive=False)
                m_recs = gr.Textbox(label='Registros',  interactive=False)
            with gr.Row():
                m_it   = gr.Textbox(label='Imposible travel', interactive=False)
                m_id   = gr.Textbox(label='Solicitudes ID',   interactive=False)
            m_tokens = gr.Textbox(label='Tokens activos', interactive=False)
            gr.Markdown('### Resolucion de identidad (autorizacion dual)')
            res_record = gr.Textbox(placeholder='Record ID', label='Registro')
            with gr.Row():
                res_auth1 = gr.Textbox(placeholder='Admin 1', label='Autorizador 1')
                res_auth2 = gr.Textbox(placeholder='Admin 2', label='Autorizador 2')
            res_btn    = gr.Button('Resolver identidad', variant='stop')
            res_result = gr.Textbox(label='Resultado', interactive=False)

    gr.Markdown('### Mapa del edificio')
    fig_bldg_out = gr.Image(label='Edificio', type='pil')
    with gr.Row():
        with gr.Column():
            gr.Markdown('### Trayectorias de tokens')
            fig_tl_out = gr.Plot()
        with gr.Column():
            gr.Markdown('### Ocupacion por espacio')
            fig_occ_out = gr.Plot()
    gr.Markdown('### Registro forense')
    table_out = gr.Dataframe(
        headers=['ID','Token','Rol','Espacio','Anomalia','Disonancia','Estado','Descripcion','Token override'],
        wrap=True)

    run_btn.click(fn=run_dashboard,
        inputs=[scenario_dd, n_days_sl, dry_run_chk, seed_sl, cfg_box],
        outputs=[fig_tl_out, fig_occ_out, fig_bldg_out,
                 m_evts, m_recs, m_it, m_id, m_tokens, res_result, table_out])
    res_btn.click(fn=do_id_resolution,
        inputs=[res_record, res_auth1, res_auth2], outputs=[res_result])

print('Dashboard listo. Lanzando...')
demo.launch(share=False, inbrowser=True)

## 8. ExportPackage y persistencia

In [ ]:
import zipfile

forensic_data = [r.to_mode2_payload() for r in FORENSIC_LOG]
with open(ARTIFACTS / 'forensic_log_movement.json', 'w') as f:
    json.dump(forensic_data, f, indent=2)

with open(ARTIFACTS / 'id_resolution_log.json', 'w') as f:
    json.dump(ID_RESOLUTION_LOG, f, indent=2)

baselines_serial = {k: v.tolist() for k, v in _badge_baselines.items()}
with open(ARTIFACTS / 'badge_baselines.json', 'w') as f:
    json.dump(baselines_serial, f, indent=2)

manifest = {
    'run_id': RUN_ID, 'module': 'MOCG-CapaMovimiento',
    'mode2_compatible': True, 'principal_id': MCFG['principal_id'],
    'bimester': MCFG['bimester'],
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'privacy_note': 'Tokens pseudoanonimos - no contiene IDs reales',
    'forensic_records': len(FORENSIC_LOG),
    'id_requests': sum(1 for r in FORENSIC_LOG if r.id_resolution_requested),
    'id_resolutions': len(ID_RESOLUTION_LOG),
    'spaces_monitored': len(MCFG['spaces']),
    'tokens_monitored': len(_badge_baselines),
}
with open(ARTIFACTS / 'run_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

export_zip = ARTIFACTS.parent.parent / f'mocg_movimiento_{MCFG["principal_id"]}.zip'
with zipfile.ZipFile(export_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in ARTIFACTS.iterdir():
        zf.write(fp, arcname=f'capa_movimiento/{fp.name}')

print(f'ExportPackage: {export_zip}')
print(f'Registros forenses : {len(FORENSIC_LOG)}')
print(f'Solicitudes ID     : {manifest["id_requests"]}')
print(f'IDs resueltos      : {manifest["id_resolutions"]}')
print(f'Nota de privacidad : {manifest["privacy_note"]}')
print(f'Tamano             : {export_zip.stat().st_size/1024:.1f} KB')